In [ ]:
# Cell 1 – Install (run once)
import subprocess
subprocess.run(["pip", "install", "torch==2.5.1", "datasets",
                "transformers", "sentencepiece", "accelerate"])

In [1]:
# Cell 2 – Imports
import warnings
warnings.filterwarnings('ignore')
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig

In [2]:
!pip install --upgrade datasets transformers

In [3]:
# Cell 3 – Load dataset
dataset = load_dataset("knkarthick/dialogsum")
example_indices = [40, 200]
dash_line = '-' * 100

In [4]:
# Cell 4 – Explore examples
for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:\n', dataset['test'][index]['dialogue'])
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:\n', dataset['test'][index]['summary'])
    print(dash_line)

----------------------------------------------------------------------------------------------------
Example  1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
 #Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
 #Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------

In [5]:
# Cell 5 – Load model & tokenizer
model_name = 'google/flan-t5-base'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [6]:
# testing tokrnizer
sentence = "what are the names of the children of isreal?"
encoded = tokenizer(sentence, return_tensors='pt')
decoded = tokenizer.decode(encoded['input_ids'][0], skip_special_tokens=True)
print('ENCODED: ', encoded['input_ids'][0])
print('DECODED: ', decoded)

ENCODED:  tensor([ 125,   33,    8, 3056,   13,    8,  502,   13,   19, 6644,   58,    1])
DECODED:  what are the names of the children of isreal?


In [7]:
# Zero shot , no promtp (bad baseline)
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']
    inputs = tokenizer(dialogue, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(inputs["input_ids"], max_new_tokens=50)[0],
        skip_special_tokens=True
    )
    print(dash_line)
    print('Example ', i +1)
    print(dash_line)
    print(f'INPUT DIALOGUE:\n{dialogue}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION > WITHOUT PROMPT ENGINEERING:\n{output}\n')

----------------------------------------------------------------------------------------------------
Example  1
----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
MODEL GENERATION > WITHOUT PROMPT ENGINEERING:
Person1: It's ten to nine.

-------------------------

In [9]:
#Zero-shot > with intruction prompt
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']
    prompt = f"""Summarize the following conversation.{dialogue}
    Summary:"""
    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(inputs["input_ids"], max_new_tokens=50)[0],
        skip_special_tokens=True
    )
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')

----------------------------------------------------------------------------------------------------
Example  1
----------------------------------------------------------------------------------------------------
INPUT PROMPT:
Summarize the following conversation.#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
    Summary:
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.
----------------------------------------------------------------------------------------------------
MODEL GENERATION - ZERO SHOT:
The train is about to 

In [11]:
# Make_prompt helper
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']
        prompt += f"""Dialogue:
{dialogue}
What was going on?
{summary}

"""
    dialogue = dataset['test'][example_index_to_summarize]['dialogue']
    prompt += f"""Dialogue:
{dialogue}
What was going on?
"""
    return prompt
        

In [13]:
#ONE SHOT PROMPTING
example_indices_full = [40]
example_index_to_summarize = 200
one_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)
print(one_shot_prompt)

summary = dataset['test'][example_index_to_summarize]['summary']
inputs = tokenizer(one_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], max_new_tokens=50)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

Dialogue:
#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.
What was going on?
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.

Dialogue:
#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a m

In [18]:
# Cell 11 – Few-shot
example_indices_full = [40, 80, 120]
few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)
inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], max_new_tokens=50)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')

----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to upgrade his hardware.


Flan-T5-base was trained with a maximum context window of 512 tokens. Your few-shot prompt with 3 full dialogue+summary examples is 818 tokens — 306 tokens over the limit.
Think of it like a form with 512 boxes, but you're trying to write 818 characters. The model just silently truncates — it reads the first 512 tokens and ignores the rest. So it may be missing part of your third example or even the dialogue you actually want summarized.

Option A — Just proceed (what the lab intends)
The warning won't crash anything. The lab instructor literally says "just ignore some of these errors" in the transcript. Run the cell, see the output, and observe that few-shot didn't improve much over one-shot — which is the lesson being taught.

Option B — Fix it properly with truncation
Replace your few-shot cell with this cleaner version that explicitly tells the tokenizer to truncate safely:

In [16]:
# FEWSHOT
# Use only 2 examples instead of 3 — stays under 512 tokens
example_indices_full = [40, 80]   # ← dropped index 120
example_index_to_summarize = 200

few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

# Check the token count BEFORE generating
token_count = len(tokenizer(few_shot_prompt)['input_ids'])
print(f'Prompt token count: {token_count} / 512')

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], max_new_tokens=50)[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT (2 examples):\n{output}')

Prompt token count: 630 / 512
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT (2 examples):
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to upgrade his hardware.


In [17]:
# Test all three properly, checking tokens each time
test_cases = {
    "ZERO SHOT": [],
    "ONE SHOT":  [40],
    "FEW SHOT (2 examples)": [40, 80],
}

example_index_to_summarize = 200
summary = dataset['test'][example_index_to_summarize]['summary']

for label, indices_full in test_cases.items():
    if indices_full:
        prompt = make_prompt(indices_full, example_index_to_summarize)
    else:
        dialogue = dataset['test'][example_index_to_summarize]['dialogue']
        prompt = f"""Summarize the following conversation.
{dialogue}
Summary:"""

    token_count = len(tokenizer(prompt)['input_ids'])
    fits = "✅" if token_count <= 512 else "⚠️ TRUNCATED"

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
    output = tokenizer.decode(
        model.generate(inputs["input_ids"], max_new_tokens=50)[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print(f'{label}  |  tokens: {token_count}/512  {fits}')
    print(dash_line)
    print(f'MODEL OUTPUT:\n{output}')

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}')

----------------------------------------------------------------------------------------------------
ZERO SHOT  |  tokens: 230/512  ✅
----------------------------------------------------------------------------------------------------
MODEL OUTPUT:
#Person1#: I'm thinking of upgrading my computer.
----------------------------------------------------------------------------------------------------
ONE SHOT  |  tokens: 391/512  ✅
----------------------------------------------------------------------------------------------------
MODEL OUTPUT:
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to add a CD-ROM drive.
----------------------------------------------------------------------------------------------------
FEW SHOT (2 examples)  |  tokens: 630/512  ⚠️ TRUNCATED
----------------------------------------------------------------------------------------------------
MODEL OUTPUT:
Person1 suggests that they should upgrade their

In [19]:
#GenerationConfig experiments
generation_config = GenerationConfig(max_new_tokens=50)
# Uncomment these to experiment:
# generation_config = GenerationConfig(max_new_tokens=10)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], generation_config=generation_config)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to upgrade his hardware.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.



In [21]:
#GenerationConfig experiments
#generation_config = GenerationConfig(max_new_tokens=50)
# Uncomment these to experiment:
generation_config = GenerationConfig(max_new_tokens=10)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], generation_config=generation_config)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
#Person1 wants to upgrade his system.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.



In [22]:
#GenerationConfig experiments
#generation_config = GenerationConfig(max_new_tokens=50)
# Uncomment these to experiment:
# generation_config = GenerationConfig(max_new_tokens=10)
generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], generation_config=generation_config)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
#Person1 wants to upgrade his system. #Person2 wants to add a painting program to his software. #Person1 wants to upgrade his hardware.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.



In [23]:
#GenerationConfig experiments
#generation_config = GenerationConfig(max_new_tokens=50)
# Uncomment these to experiment:
# generation_config = GenerationConfig(max_new_tokens=10)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], generation_config=generation_config)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
Person1 is considering upgrading his system. He has no idea what he should do. He might need a painting program. He also needs a faster processor, memory and a faster modem.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.



In [24]:
#GenerationConfig experiments
#generation_config = GenerationConfig(max_new_tokens=50)
# Uncomment these to experiment:
# generation_config = GenerationConfig(max_new_tokens=10)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(inputs["input_ids"], generation_config=generation_config)[0],
    skip_special_tokens=True
)
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')

----------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
#Person2 doesn't know what he'll need a certain amount to upgrade his computer. He considers adding any graphics and programs but does not want to do anything with a painting program.
----------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.

